In [1]:
import pandas as pd
import numpy as np
from scipy import stats
from utils.utils import setup_project_root

setup_project_root()

Connecting to SMTP server...
Sending email to alice@example.com: [ALERT] Your server is melting down!
Successfully notified Alice
Connecting to Twilo...
Sending SMS to 555-0199: [ALERT] [ALERT] Your server is melting down!
Successfully notified Alice


WindowsPath('C:/Users/zak/Projects/PyCharmProjects/data-science')

In [37]:
class DataScienceProject():
    """
    This class represents a Data Science project. The project will contain 1 or more experiments that lead to a predictive model.
    """
    def __init__(self, name):
        """

        :param name: The name of the project, usually the
        """
        self.name = name

In [ ]:
class Experiment():
    def __init__(self, name, ):
        self.name = name


    def result(self):
        pass

    def experiment_pipeline(self):
        pass

    def input(self):
        """
        The experiment will take a dataset and pipeline.
        :return:
        """
        pass



In [2]:
diabetes = pd.read_csv(r'data/diabetes.csv')

In [19]:
diabetes.isna().mean().sort_values(ascending=False)

Serum_Insulin        0.486979
Skin_Fold            0.295573
Diastolic_BP         0.045573
BMI                  0.014323
Glucose              0.006510
Pregnant             0.000000
Diabetes_Pedigree    0.000000
Age                  0.000000
Class                0.000000
dtype: float64

In [6]:
import plotly.graph_objects as go
import plotly.express as px

In [21]:
px.bar(diabetes.isna().mean().sort_values(), orientation='h')

Imputation Analysis

Imputations relies on the dimensionality of the dat and the mechanism for missingness.

In [22]:
diabetes_imputed = pd.read_csv(r'data/diabetes_imputed_sota.csv')

In [23]:
px.bar(diabetes_imputed.isna().mean().sort_values(), orientation='h')

In [25]:
import pandas as pd

# Load data
df = pd.read_csv(r'data/diabetes.csv')

# 1. Identify columns with missing values
missing_cols = df.columns[df.isnull().any()].tolist()

# 2. Construct the binary indicator matrix M
M = df[missing_cols].isnull().astype(int)
M = M.add_suffix('_missing')

# 3. Concatenate observed features and M to compute the cross-correlation matrix
combined_df = pd.concat([df.drop(columns=missing_cols), M], axis=1)
corr_matrix = combined_df.corr()

# 4. Extract correlations between M and X_obs
for col in missing_cols:
    target_m = f"{col}_missing"
    # Isolate correlations with observed features, dropping M-to-M correlations
    correlations = corr_matrix[target_m].drop([f"{c}_missing" for c in missing_cols])
    print(f"Top correlations for {target_m}:")
    print(correlations.abs().sort_values(ascending=False).head(3))

Top correlations for Glucose_missing:
Age                  0.031966
Pregnant             0.025123
Diabetes_Pedigree    0.022413
Name: Glucose_missing, dtype: float64
Top correlations for Diastolic_BP_missing:
Diabetes_Pedigree    0.055071
Class                0.049597
Age                  0.046977
Name: Diastolic_BP_missing, dtype: float64
Top correlations for Skin_Fold_missing:
Age                  0.221029
Diabetes_Pedigree    0.153738
Pregnant             0.152681
Name: Skin_Fold_missing, dtype: float64
Top correlations for Serum_Insulin_missing:
Age                  0.211885
Pregnant             0.170156
Diabetes_Pedigree    0.166357
Name: Serum_Insulin_missing, dtype: float64
Top correlations for BMI_missing:
Class                0.042271
Age                  0.028579
Diabetes_Pedigree    0.014054
Name: BMI_missing, dtype: float64


## Exploratory Data Analysis

1. Missing Data Analysis
    - Missingness Analysis
    - Missing Data Imputation
2. Duplicate Data
3. Univariate Analysis
4. Bivariate Analysis
5. Multivariate Analysis
6. Handling Outliers

In [33]:
class MissingDataTutor:
    # I'm creating this class to formalise my knowledge on handling missing data. Only I will use the class. The tutor name comes from the fact that I want the object to act as a tutor for me when I use it. Reminding me of the logic I created here. I expect the printed output to be overly verbose in that regard, not the usual coding format.

    def __init__(self):
        self._df = None
        self._missing_cols = None
        self._mechanism = {}   # col -> 'MCAR' | 'MAR'
        self._df_imputed = None

    def missingness(self, df: pd.DataFrame, correlation_threshold: float = 0.15):
        self._df = df.copy()
        self._missing_cols = df.columns[df.isnull().any()].tolist()

        # ── Overview ────────────────────────────────────────────────────────────
        print("=" * 70)
        print("MISSING DATA ANALYSIS")
        print("=" * 70)
        print(f"\nDataset: {len(df)} rows × {len(df.columns)} columns")
        print(f"Columns with missing data: {len(self._missing_cols)}\n")

        # ── Step 1: Missingness rates ────────────────────────────────────────────
        rates = df.isnull().mean().sort_values(ascending=False)
        print("─" * 50)
        print("STEP 1: Missingness Rates")
        print("─" * 50)
        print("""
We first ask: how much data is missing, and from which columns?

    Missing rate = (number of NaN values) / (total rows)

A high missing rate (> 40%) may warrant dropping the column entirely.
Between 5–40% is the typical imputation target zone.
""")
        for col in self._missing_cols:
            n = df[col].isnull().sum()
            pct = rates[col] * 100
            flag = "  ← high, consider dropping" if pct > 40 else ""
            print(f"  {col}: {n} missing ({pct:.1f}%){flag}")

        px.bar(
            rates[rates > 0].sort_values(),
            orientation='h',
            title="Missing Data Rates by Column",
            labels={"value": "Missing Rate", "index": "Column"},
        ).show()

        # ── Step 2: Diagnose missingness mechanism ───────────────────────────────
        print()
        print("─" * 50)
        print("STEP 2: Diagnosing the Missingness Mechanism")
        print("─" * 50)
        print("""
Missing data falls into three categories — each demands a different response:

  MCAR — Missing Completely At Random
      The probability of a value being missing is unrelated to any data,
      observed or unobserved. Like a random hardware fault.
      → Simple imputation (median) or listwise deletion is statistically valid.

  MAR  — Missing At Random
      Missingness depends on *observed* features, but not on the missing value
      itself. E.g., older patients are less likely to have insulin recorded,
      but among any age group, who is missing is random.
      → Model-based imputation (MICE / IterativeImputer) is appropriate.

  MNAR — Missing Not At Random
      Missingness depends on the unobserved value itself. E.g., patients with
      very high glucose are less likely to have it recorded.
      → Cannot be resolved from the data alone; needs domain knowledge or
        sensitivity analysis. We can flag suspicion but cannot confirm from data.

HOW WE DIAGNOSE:
  Build a binary indicator matrix M, where M_ij = 1 if X_ij is missing.
  Correlate each column of M with the *observed* features X_obs.
  High correlation with observed features → evidence for MAR (or MNAR).
  Low correlation everywhere → consistent with MCAR (not proof, but plausible).
""")
        M = df[self._missing_cols].isnull().astype(int).add_suffix('_missing')
        combined = pd.concat([df.drop(columns=self._missing_cols), M], axis=1)
        corr = combined.corr()

        print(f"Correlation threshold for MAR classification: |r| ≥ {correlation_threshold}\n")

        for col in self._missing_cols:
            target = f"{col}_missing"
            obs_corrs = corr[target].drop(
                [f"{c}_missing" for c in self._missing_cols], errors='ignore'
            )
            top3 = obs_corrs.abs().sort_values(ascending=False).head(3)
            max_corr = top3.iloc[0]

            if max_corr >= correlation_threshold:
                mechanism = "MAR"
                verdict = f"Max |r| = {max_corr:.3f} ≥ {correlation_threshold} → MAR likely."
            else:
                mechanism = "MCAR"
                verdict = f"Max |r| = {max_corr:.3f} < {correlation_threshold} → consistent with MCAR."

            self._mechanism[col] = mechanism
            print(f"  {col}  [{mechanism}]")
            print(f"    Top correlations with observed features: {dict(top3.round(3))}")
            print(f"    {verdict}\n")

        # ── Step 3: Recommendation ───────────────────────────────────────────────
        print("─" * 50)
        print("STEP 3: Recommendation")
        print("─" * 50)

        mar_cols  = [c for c, m in self._mechanism.items() if m == "MAR"]
        mcar_cols = [c for c, m in self._mechanism.items() if m == "MCAR"]

        if mar_cols:
            print(f"\nColumns likely MAR: {mar_cols}")
            print("""  → Use IterativeImputer (MICE algorithm).
    MICE models each missing column as a regression on all other columns,
    iterates until convergence, and respects inter-variable relationships.""")

        if mcar_cols:
            print(f"\nColumns consistent with MCAR: {mcar_cols}")
            print("""  → Use median imputation (SimpleImputer).
    Since missingness is random, this is unbiased and fast.
    Median is preferred over mean when outliers are present.""")

        print("\nCall .impute() to apply the recommended strategy.\n")

    def impute(self):
        if self._df is None:
            print("Run .missingness(df) first.")
            return

        df = self._df.copy()
        mar_cols  = [c for c, m in self._mechanism.items() if m == "MAR"]
        mcar_cols = [c for c, m in self._mechanism.items() if m == "MCAR"]

        print("=" * 70)
        print("IMPUTATION")
        print("=" * 70)

        # ── MCAR: median imputation ──────────────────────────────────────────────
        if mcar_cols:
            print(f"\nMedian imputation → {mcar_cols}")
            print("""
SimpleImputer(strategy='median') replaces each NaN with the column median.
For MCAR data this is unbiased. We fill these first so that IterativeImputer
(if needed) can use them as predictors without treating them as targets.
""")
            from sklearn.impute import SimpleImputer
            df[mcar_cols] = SimpleImputer(strategy='median').fit_transform(df[mcar_cols])

        # ── MAR: IterativeImputer (MICE) ─────────────────────────────────────────
        if mar_cols:
            print(f"\nIterativeImputer (MICE) → {mar_cols}")
            print("""
IterativeImputer implements MICE (Multiple Imputation by Chained Equations).
For each missing column in turn it fits a BayesianRidge regression using all
other columns as predictors, fills in the missing values, then moves to the
next column. The full pass repeats max_iter times until imputations converge.

Because it conditions on all other columns it correctly handles MAR data where
missingness is driven by observed features.
""")
            from sklearn.experimental import enable_iterative_imputer  # noqa
            from sklearn.impute import IterativeImputer
            df[:] = IterativeImputer(max_iter=10, random_state=0).fit_transform(df)

        # ── Verification ─────────────────────────────────────────────────────────
        remaining = df.isnull().sum().sum()
        print("─" * 50)
        print("VERIFICATION")
        print("─" * 50)
        if remaining == 0:
            print("\n✓ No missing values remain. Imputation complete.\n")
        else:
            print(f"\n⚠  {remaining} missing values remain. Investigate further.\n")

        self._df_imputed = df

        # ── Visualisation ────────────────────────────────────────────────────────
        print("─" * 50)
        print("VISUALISATION")
        print("─" * 50)
        print("""
For each imputed column:
  Scatter — shows where the imputed points (×) sit among observed points (·).
            A good imputation fills gaps without creating obvious outliers.
  Histogram — compares the distribution of observed values vs imputed values.
              The imputed distribution should broadly mirror the observed one.
""")
        for col in mcar_cols + mar_cols:
            was_missing = self._df[col].isnull()

            fig = go.Figure()
            fig.add_trace(go.Scatter(
                x=self._df.index[~was_missing],
                y=self._df[col][~was_missing],
                mode='markers',
                name='Observed',
                marker=dict(color='steelblue', opacity=0.5),
            ))
            fig.add_trace(go.Scatter(
                x=self._df.index[was_missing],
                y=df[col][was_missing],
                mode='markers',
                name='Imputed',
                marker=dict(color='crimson', symbol='x', size=9),
            ))
            fig.update_layout(
                title=f"{col}: Observed vs Imputed Values",
                xaxis_title="Row Index",
                yaxis_title=col,
            )
            fig.show()

            fig2 = go.Figure()
            fig2.add_trace(go.Histogram(
                x=self._df[col].dropna(), name='Observed', opacity=0.6,
                marker_color='steelblue',
            ))
            fig2.add_trace(go.Histogram(
                x=df[col][was_missing], name='Imputed', opacity=0.7,
                marker_color='crimson',
            ))
            fig2.update_layout(
                barmode='overlay',
                title=f"{col}: Distribution — Observed vs Imputed",
                xaxis_title=col,
                yaxis_title="Count",
            )
            fig2.show()

        return df


In [34]:
missing_data_tutor = MissingDataTutor()

In [35]:
missing_data_tutor.missingness(df)

MISSING DATA ANALYSIS

Dataset: 768 rows × 9 columns
Columns with missing data: 5

──────────────────────────────────────────────────
STEP 1: Missingness Rates
──────────────────────────────────────────────────

We first ask: how much data is missing, and from which columns?

    Missing rate = (number of NaN values) / (total rows)

A high missing rate (> 40%) may warrant dropping the column entirely.
Between 5–40% is the typical imputation target zone.

  Glucose: 5 missing (0.7%)
  Diastolic_BP: 35 missing (4.6%)
  Skin_Fold: 227 missing (29.6%)
  Serum_Insulin: 374 missing (48.7%)  ← high, consider dropping
  BMI: 11 missing (1.4%)



──────────────────────────────────────────────────
STEP 2: Diagnosing the Missingness Mechanism
──────────────────────────────────────────────────

Missing data falls into three categories — each demands a different response:

  MCAR — Missing Completely At Random
      The probability of a value being missing is unrelated to any data,
      observed or unobserved. Like a random hardware fault.
      → Simple imputation (median) or listwise deletion is statistically valid.

  MAR  — Missing At Random
      Missingness depends on *observed* features, but not on the missing value
      itself. E.g., older patients are less likely to have insulin recorded,
      but among any age group, who is missing is random.
      → Model-based imputation (MICE / IterativeImputer) is appropriate.

  MNAR — Missing Not At Random
      Missingness depends on the unobserved value itself. E.g., patients with
      very high glucose are less likely to have it recorded.
      → Cannot be resolved from the 

In [36]:
missing_data_tutor.impute()

IMPUTATION

Median imputation → ['Glucose', 'Diastolic_BP', 'BMI']

SimpleImputer(strategy='median') replaces each NaN with the column median.
For MCAR data this is unbiased. We fill these first so that IterativeImputer
(if needed) can use them as predictors without treating them as targets.


IterativeImputer (MICE) → ['Skin_Fold', 'Serum_Insulin']

IterativeImputer implements MICE (Multiple Imputation by Chained Equations).
For each missing column in turn it fits a BayesianRidge regression using all
other columns as predictors, fills in the missing values, then moves to the
next column. The full pass repeats max_iter times until imputations converge.

Because it conditions on all other columns it correctly handles MAR data where
missingness is driven by observed features.

──────────────────────────────────────────────────
VERIFICATION
──────────────────────────────────────────────────

✓ No missing values remain. Imputation complete.

────────────────────────────────────────────────

,Pregnant,Glucose,Diastolic_BP,Skin_Fold,Serum_Insulin,BMI,Diabetes_Pedigree,Age,Class
0,6.0,148.0,72.0,35.000000,219.011897,33.6,0.627,50,1.0
1,1.0,85.0,66.0,29.000000,70.352494,26.6,0.351,31,0.0
2,8.0,183.0,64.0,21.539615,268.035220,23.3,0.672,32,1.0
3,1.0,89.0,66.0,23.000000,94.000000,28.1,0.167,21,0.0
4,0.0,137.0,40.0,35.000000,168.000000,43.1,2.288,33,1.0
...,...,...,...,...,...,...,...,...,...
763,10.0,101.0,76.0,48.000000,180.000000,32.9,0.171,63,0.0
764,2.0,122.0,70.0,27.000000,158.511050,36.8,0.340,27,0.0
765,5.0,121.0,72.0,23.000000,112.000000,26.2,0.245,30,0.0
766,1.0,126.0,60.0,27.900442,173.604941,30.1,0.349,47,1.0
